# ☕ KapsamKafe — Viral Radar & Caption Agent

**İki modül:**
- 📡 **Viral Radar** — TikTok'tan hashtag/kullanıcı/trending/arama bazlı video verisi çeker, viral skor hesaplar
- 🎙️ **Caption Agent** — faster-whisper ile Türkçe SRT üretir, yabancı dilli videolarda NLLB-200 ile otomatik çevirir
- 🎬 **Bonus** — FFmpeg ile altyazıyı videoya gömer

> **Runtime:** `Runtime → Change runtime type → T4 GPU` seç (Caption Agent için önemli)  
> **Token:** `MS_TOKEN` boş bırakılabilir — sorun çıkarsa aşağıdaki talimatı takip et

---
## 0. Kurulum

In [ ]:
# Tüm bağımlılıkları yükle
!pip install -q TikTokApi playwright faster-whisper transformers sentencepiece ffmpeg-python
!python -m playwright install chromium
!apt-get install -y -q ffmpeg 2>/dev/null || echo 'ffmpeg zaten yüklü'
import os
os.makedirs('output', exist_ok=True)
print('✅ Kurulum tamamlandı')

---
## 1. Yapılandırma

**msToken alma (gerekirse):**
1. Bilgisayardan `tiktok.com`'a giriş yap
2. F12 → Application → Cookies → `msToken` değerini kopyala
3. Aşağıdaki `MS_TOKEN = ""` satırına yapıştır

**Not:** Boş bırakılırsa hashtag/kullanıcı modları genellikle çalışır.

In [ ]:
# ─── AYARLAR ───────────────────────────────────────────────────────────────
MS_TOKEN = ""   # Boş bırak, hata alırsan tiktok.com'dan kopyala

# Viral Radar varsayılan modu: 'hashtag' | 'user' | 'trending' | 'search'
DEFAULT_MODE  = "hashtag"
DEFAULT_QUERY = "kahve"        # hashtag/user/search için kullanılır
VIDEO_COUNT   = 30             # çekilecek video sayısı

# Caption Agent
WHISPER_MODEL = "medium"       # tiny/base/small/medium/large-v3
WHISPER_LANG  = "tr"           # None → otomatik algıla
TRANSLATE_TO_TR = True         # Türkçe değilse otomatik çevir
# ───────────────────────────────────────────────────────────────────────────
print('✅ Yapılandırma yüklendi')

---
## 2. 📡 Viral Radar

In [ ]:
import asyncio, json, time, math
from datetime import datetime, timezone
from TikTokApi import TikTokApi

# ─── Viral skor hesaplama ───────────────────────────────────────────────────
def viral_score(video: dict) -> float:
    """
    Skor = (likes + comments*2 + shares*3) / max(views, 1)
    Tazelik bonusu: 24 saatten yeni → 1.5x, 72 saatten yeni → 1.2x
    """
    stats = video.get("stats", {})
    likes    = stats.get("diggCount", 0)
    comments = stats.get("commentCount", 0)
    shares   = stats.get("shareCount", 0)
    views    = max(stats.get("playCount", 1), 1)

    engagement = (likes + comments * 2 + shares * 3) / views

    create_time = video.get("createTime", 0)
    age_hours = (time.time() - create_time) / 3600 if create_time else 999
    if age_hours < 24:
        freshness = 1.5
    elif age_hours < 72:
        freshness = 1.2
    else:
        freshness = 1.0

    return round(engagement * freshness * 100, 4)


def parse_video(v) -> dict:
    """TikTokApi video nesnesinden temiz sözlük üretir."""
    raw = v.as_dict if hasattr(v, 'as_dict') else v
    stats = raw.get("stats", {})
    author = raw.get("author", {})
    music  = raw.get("music", {})
    return {
        "id":          raw.get("id"),
        "desc":        raw.get("desc", ""),
        "createTime":  raw.get("createTime"),
        "author":      author.get("uniqueId", ""),
        "views":       stats.get("playCount", 0),
        "likes":       stats.get("diggCount", 0),
        "comments":    stats.get("commentCount", 0),
        "shares":      stats.get("shareCount", 0),
        "hashtags":    [h["hashtagName"] for h in raw.get("challenges", [])],
        "music_title": music.get("title", ""),
        "music_author":music.get("authorName", ""),
        "viral_score": viral_score(raw),
    }


# ─── Ana çekme fonksiyonu ───────────────────────────────────────────────────
async def fetch_videos(mode: str, query: str = "", count: int = 30) -> list:
    ms_token = MS_TOKEN if MS_TOKEN.strip() else None
    kwargs   = {"ms_tokens": [ms_token]} if ms_token else {}

    videos = []
    async with TikTokApi(**kwargs) as api:
        await api.create_sessions(
            ms_tokens=[ms_token] if ms_token else [],
            num_sessions=1,
            sleep_after=3,
            headless=True,
        )

        if mode == "trending":
            async for v in api.trending.videos(count=count):
                videos.append(parse_video(v))

        elif mode == "hashtag":
            tag = api.hashtag(name=query)
            async for v in tag.videos(count=count):
                videos.append(parse_video(v))

        elif mode == "user":
            user = api.user(username=query)
            async for v in user.videos(count=count):
                videos.append(parse_video(v))

        elif mode == "search":
            async for v in api.search.videos(query, count=count):
                videos.append(parse_video(v))

        else:
            raise ValueError(f"Geçersiz mod: {mode}. Seçenekler: trending, hashtag, user, search")

    return sorted(videos, key=lambda x: x["viral_score"], reverse=True)


print('✅ Viral Radar fonksiyonları yüklendi')

In [ ]:
# ─── Çalıştır ──────────────────────────────────────────────────────────────
print(f'🔍 Mod: {DEFAULT_MODE}  |  Sorgu: "{DEFAULT_QUERY}"  |  Sayı: {VIDEO_COUNT}')

try:
    results = await fetch_videos(DEFAULT_MODE, DEFAULT_QUERY, VIDEO_COUNT)
except Exception as e:
    print(f'⚠️  Hata: {e}')
    if 'EmptyResponseException' in str(type(e)) or 'msToken' in str(e).lower():
        print('👉 msToken gerekebilir — tiktok.com\'dan kopyala ve MS_TOKEN değişkenine yapıştır')
    results = []

if results:
    ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
    safe_q   = DEFAULT_QUERY.replace(' ', '_')[:30]
    out_path = f'output/viral_radar_{DEFAULT_MODE}_{safe_q}_{ts}.json'

    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f'\n✅ {len(results)} video kaydedildi → {out_path}')
    print(f'\n🏆 TOP 5 Viral Skor:')
    for i, v in enumerate(results[:5], 1):
        print(f'  {i}. [{v["viral_score"]:>8.4f}] @{v["author"]} | 👁 {v["views"]:,} | {v["desc"][:60]}')
else:
    print('Sonuç bulunamadı.')

---
## 3. 🎙️ Caption Agent

faster-whisper ile SRT üretir. Kaynak dil Türkçe değilse NLLB-200 ile otomatik çevirir.

In [ ]:
import torch
from faster_whisper import WhisperModel
from transformers import pipeline as hf_pipeline
from pathlib import Path

# Whisper dil kodu → FLORES-200 kodu eşlemesi
LANG_TO_FLORES = {
    'en': 'eng_Latn', 'ar': 'arb_Arab', 'de': 'deu_Latn',
    'fr': 'fra_Latn', 'es': 'spa_Latn', 'ru': 'rus_Cyrl',
    'it': 'ita_Latn', 'pt': 'por_Latn', 'ja': 'jpn_Jpan',
    'ko': 'kor_Hang', 'zh': 'zho_Hans', 'hi': 'hin_Deva',
    'fa': 'pes_Arab', 'nl': 'nld_Latn', 'tr': 'tur_Latn',
}

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
compute   = 'float16' if device == 'cuda' else 'int8'
print(f'⚙️  Device: {device} | Compute: {compute}')

# ─── Modelleri yükle ───────────────────────────────────────────────────────
print(f'📥 Whisper {WHISPER_MODEL} yükleniyor...')
whisper = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute)
print('✅ Whisper hazır')

nllb = None
def get_nllb():
    global nllb
    if nllb is None:
        print('📥 NLLB-200 yükleniyor (ilk kez ~2 dk)...')
        nllb = hf_pipeline(
            'translation',
            model='facebook/nllb-200-distilled-600M',
            device=0 if device == 'cuda' else -1,
            max_length=512,
        )
        print('✅ NLLB-200 hazır')
    return nllb


# ─── Zaman biçimlendirme ────────────────────────────────────────────────────
def fmt_time(seconds: float) -> str:
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


# ─── Segment çevirisi ───────────────────────────────────────────────────────
def translate_segment(text: str, src_lang: str) -> str:
    src_flores = LANG_TO_FLORES.get(src_lang)
    if not src_flores or src_lang == 'tr':
        return text
    translator = get_nllb()
    out = translator(
        text,
        src_lang=src_flores,
        tgt_lang='tur_Latn',
    )
    return out[0]['translation_text']


# ─── Ana fonksiyon: video → SRT ────────────────────────────────────────────
def transcribe_to_srt(
    video_path: str,
    output_srt: str = None,
    language:   str = None,
    translate:  bool = True,
) -> str:
    """
    video_path : yerel video/ses dosyası
    output_srt : çıktı SRT yolu (None → otomatik)
    language   : 'tr', 'en' vb. (None → otomatik algıla)
    translate  : Türkçe değilse NLLB-200 ile çevir
    """
    if output_srt is None:
        stem = Path(video_path).stem
        output_srt = f'output/{stem}_tr.srt'

    print(f'🎙️  Transkripsiyon: {video_path}')
    segments, info = whisper.transcribe(
        video_path,
        language=language,
        word_timestamps=True,
        beam_size=5,
    )
    detected_lang = info.language
    print(f'   Tespit edilen dil: {detected_lang} (güven: {info.language_probability:.2%})')

    lines   = []
    seg_idx = 0
    for seg in segments:
        seg_idx += 1
        text = seg.text.strip()
        if translate and detected_lang != 'tr':
            text = translate_segment(text, detected_lang)

        lines.append(str(seg_idx))
        lines.append(f'{fmt_time(seg.start)} --> {fmt_time(seg.end)}')
        lines.append(text)
        lines.append('')

    srt_content = '\n'.join(lines)
    with open(output_srt, 'w', encoding='utf-8') as f:
        f.write(srt_content)

    print(f'✅ SRT kaydedildi → {output_srt}  ({seg_idx} segment)')
    return output_srt


print('✅ Caption Agent fonksiyonları yüklendi')

In [ ]:
# ─── Kullanım ──────────────────────────────────────────────────────────────
# VIDEO_PATH değişkenini kendi video dosyana göre düzenle
VIDEO_PATH = '/content/sample_video.mp4'   # ← buraya kendi dosyanı yaz

if not Path(VIDEO_PATH).exists():
    print(f'⚠️  Dosya bulunamadı: {VIDEO_PATH}')
    print('   Colab sol panelinden dosya yükle veya yolu düzelt')
else:
    srt_path = transcribe_to_srt(
        video_path=VIDEO_PATH,
        language=WHISPER_LANG if WHISPER_LANG else None,
        translate=TRANSLATE_TO_TR,
    )
    print(f'\n📄 SRT önizleme (ilk 5 segment):')
    with open(srt_path, encoding='utf-8') as f:
        lines = f.readlines()
    print(''.join(lines[:20]))

---
## 4. 🎬 Bonus — FFmpeg ile Altyazı Gömme

Bebas Neue font, beyaz metin + gold outline (Mozalp tasarım sistemine uygun).

In [ ]:
# Bebas Neue font indir
!wget -q -O /tmp/BebasNeue-Regular.ttf \
  'https://github.com/dharmatype/Bebas-Neue/raw/master/fonts/BebasNeue(2019)ByDhamraType/ttf/BebasNeue-Regular.ttf' \
  && echo '✅ Bebas Neue hazır' || echo 'Font indirilemedi, sistem fontuna geçiliyor'

In [ ]:
import subprocess

FONT_PATH = '/tmp/BebasNeue-Regular.ttf'
if not Path(FONT_PATH).exists():
    FONT_PATH = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'

def burn_subtitles(
    video_path: str,
    srt_path:   str,
    output_path: str = None,
    font_size: int  = 22,
) -> str:
    """
    SRT'yi video üzerine yazar.
    Beyaz metin, gold (#FFD700) outline — Mozalp stil.
    """
    if output_path is None:
        stem = Path(video_path).stem
        output_path = f'output/{stem}_subtitled.mp4'

    style = (
        f"FontName=Bebas Neue,"
        f"FontSize={font_size},"
        f"PrimaryColour=&H00FFFFFF,"
        f"OutlineColour=&H0000D7FF,"
        f"BackColour=&H80000000,"
        f"Bold=1,"
        f"Outline=2,"
        f"Shadow=1,"
        f"Alignment=2,"
        f"MarginV=30"
    )

    cmd = [
        'ffmpeg', '-y',
        '-i', video_path,
        '-vf', f"subtitles={srt_path}:force_style='{style}':fontsdir=/tmp",
        '-c:a', 'copy',
        '-c:v', 'libx264',
        '-crf', '23',
        '-preset', 'fast',
        output_path
    ]

    print(f'🎬 Altyazı gömülüyor...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('FFmpeg hata:', result.stderr[-500:])
    else:
        print(f'✅ Tamamlandı → {output_path}')
    return output_path


print('✅ burn_subtitles fonksiyonu hazır')

In [ ]:
# ─── Kullanım ──────────────────────────────────────────────────────────────
# srt_path ve VIDEO_PATH önceki hücreden geliyor
# İkisi de tanımlıysa çalıştır
try:
    if Path(VIDEO_PATH).exists() and Path(srt_path).exists():
        final_video = burn_subtitles(VIDEO_PATH, srt_path)
    else:
        print('⚠️  Önce Caption Agent hücresini çalıştır')
except NameError:
    print('⚠️  Önce Caption Agent hücresini çalıştır (srt_path tanımlı değil)')

---
## 5. 🔗 Groq/Llama Pipeline Entegrasyonu

Viral Radar çıktısını veya SRT içeriğini Groq'a gönder.

In [ ]:
# Groq ile KapsamKafe Agent entegrasyonu
# pip install -q groq  →  ihtiyaç olursa hücreyi çalıştır

def viral_data_to_prompt(viral_json_path: str) -> str:
    """JSON viral raporu → Groq/Llama için sistem promptu"""
    with open(viral_json_path, encoding='utf-8') as f:
        data = json.load(f)

    top10 = data[:10]
    lines = []
    for i, v in enumerate(top10, 1):
        lines.append(
            f"{i}. @{v['author']} | skor:{v['viral_score']} | "
            f"görüntülenme:{v['views']:,} | "
            f"açıklama: {v['desc'][:80]} | "
            f"etiketler: {', '.join(v['hashtags'][:5])}"
        )

    return (
        "Aşağıdaki TikTok trend verileri KapsamKafe için Viral Radar tarafından toplandı.\n"
        "TOP 10 viral video:\n" + "\n".join(lines) +
        "\n\nBu veriye dayanarak KapsamKafe için içerik fikirleri öner."
    )


def srt_to_prompt(srt_path: str, max_chars: int = 3000) -> str:
    """SRT dosyasını Groq/Llama için temiz metin bloğuna çevirir"""
    with open(srt_path, encoding='utf-8') as f:
        content = f.read()
    # Timestamp ve index satırlarını temizle
    lines = [
        l.strip() for l in content.splitlines()
        if l.strip() and not l.strip().isdigit() and '-->' not in l
    ]
    text = ' '.join(lines)[:max_chars]
    return f"Aşağıdaki TikTok video transkripti için başlık ve içerik önerileri yap:\n\n{text}"


# Örnek kullanım (Groq API key gerektirir)
# from groq import Groq
# client = Groq(api_key="GROQ_API_KEY")
# prompt = viral_data_to_prompt('output/viral_radar_hashtag_kahve_....json')
# response = client.chat.completions.create(
#     model='llama3-70b-8192',
#     messages=[{'role': 'user', 'content': prompt}]
# )
# print(response.choices[0].message.content)

print('✅ Pipeline entegrasyon fonksiyonları hazır')
print('   viral_data_to_prompt(json_path) → Groq prompt')
print('   srt_to_prompt(srt_path) → Groq prompt')

---
## Notlar

| Konu | Detay |
|------|-------|
| TikTok token | msToken ~1-2 saat geçerli; sorun çıkarsa tazele |
| GPU | T4 önerilir; CPU'da medium Whisper yavaş |
| NLLB-200 | İlk yükleme ~2 dk, sonraki çalıştırmalarda önbellekte |
| SRT format | Groq/Llama'ya `srt_to_prompt()` ile temiz metin olarak gönder |
| Çıktı klasörü | Tüm dosyalar `output/` altına kaydedilir |